# Engine: Sigma Expansion — J_red/J_blue Balance Curve

**Claim:** P_red(sigma) = |J_red(sigma)|^2 / (|J_red(sigma)|^2 + |J_blue(sigma)|^2) is
NOT constant across sigma (raw |J_red|^2+|J_blue|^2 has a genuine minimum at sigma=1/2,
not a flat quantum-probability-style conservation) -- but its deviation from 1/2 near the
balance point is derivable in closed form, not fit:

```
P_red(sigma) - 1/2  ~  c1*d + c3*d^3     (d = sigma - 1/2)
```

c1, c3 come from moments of the underlying Dirichlet-style projection (M_n, L_n,
evaluated once at sigma=1/2), via product/quotient-rule differentiation of
F(sigma) = |N(sigma)|^2/D(sigma)^2. Verified against direct computation:
residual ~1e-6 near sigma=1/2, growing smoothly toward the edges of the tested
range exactly as expected for a third-order Taylor truncation.

**Origin:** 2026-07-11, testing whether Nick Lucid's (Science Asylum) quantum-state
normalization argument applies to J_red/J_blue. It does not apply directly (the raw
sum isn't conserved) -- this engine is what came from asking a sharper question of
the normalized version instead.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../../..'))
from ValaQuenta.modules.sigma_expansion import maths as sx
print('sigma_expansion module loaded')

## The raw, unnormalized question — is |J_red|^2+|J_blue|^2 constant across sigma?

In [ ]:
text = 'O Captain My Captain'
for s10 in range(1, 20, 2):
    sigma = s10 / 20.0
    r = sx.project_at_sigma(text, sigma)
    b = sx.project_at_sigma(text, 1.0 - sigma)
    raw = sum(abs(x)**2 for x in r) + sum(abs(x)**2 for x in b)
    print(f'sigma={sigma:.2f}  |Jred|^2+|Jblue|^2 = {raw:.6f}')

print()
print('NOT constant -- minimum at sigma=0.5, symmetric, grows toward the extremes.')
print('This is the honest negative result: no flat quantum-style conservation here.')

## The moments — the only inputs the closed-form derivation needs

In [ ]:
m = sx.moments(text)
print(f"M0={m['M0']:.6f}  M1={m['M1']:.6f}  M2={m['M2']:.6f}  M3={m['M3']:.6f}")
print()
print('Per-channel L0..L3 (first 3 of 16 prime channels):')
for p in [2, 3, 5]:
    L0, L1, L2, L3 = m['L_by_channel'][p]
    print(f'  p={p:2d}  L0={L0:.4f}  L1={L1:.4f}  L2={L2:.4f}  L3={L3:.4f}')

## Derived (not fitted) c1, c3

In [ ]:
coeffs = sx.taylor_coefficients(text)
print(f"c1 = {coeffs['c1']:.6f}")
print(f"c3 = {coeffs['c3']:.6f}")
print()
print('Derived via Taylor expansion of F(sigma)=|N(sigma)|^2/D(sigma)^2 at sigma=1/2,')
print('summed across all 16 prime channels. Full algebra in wiki/sigma_expansion.md.')

## Verification — predicted vs. directly-computed, across three independent test strings

In [ ]:
for text in ['O Captain My Captain', 'RSA private key recovery', 'zero divisor']:
    result = sx.verify_against_actual(text)
    print(f"{text!r}:  c1={result['coefficients']['c1']:.6f}  c3={result['coefficients']['c3']:.6f}"
          f"  max_residual={result['max_residual']:.6f}  confidence={result['confidence']}")

## What this is, and isn't

**Is:** a real, verified closed-form derivation for the specific `i^-sigma` Dirichlet-
projection construction used in `SedenionSpectralRelativity/layer_spectrograph.py` and
this session's bispectrum work. A genuine error-check tool for that construction: cheap
(one pass over moments) vs. expensive (a fresh O(N) sweep per sigma queried).

**Isn't:** directly applicable to `VAPMIP/monad.py`'s Engine, which computes sigma via
`_word_zero_idx` (prime hash -> address) and `_gamma_at` (Newton's method on the real
Riemann zeta function's zeros) -- a genuinely different mechanism. Using this engine to
error-check the monad specifically would require re-deriving the same Taylor-expansion
method against *that* engine's actual sigma formula, not substituting this one in.